# Lab 3 — Orchestrate Foundry Agents with Python

## Scenario

In Labs 1 and 2 you built two AI agents in the Microsoft Foundry portal:

| Agent | Model | Role |
|-------|-------|------|
| **ZavaGroceriesInventoryAgent** | gpt-5-mini | CRUD operations on live store data (daily financials + hourly sales) via MCP tools |
| **SalesInsightsAgent** | gpt-4o | Analyzes store performance data + news context, produces structured recommendations |

In this lab you will:
1. **Connect** to your Foundry project from Python
2. **Test each agent individually** by sending prompts and viewing responses
3. **Chain both agents in a workflow** — SalesInsightsAgent produces recommendations, then ZavaGroceriesInventoryAgent acts on them

> **Key concept:** Agent behavior (instructions, tools, model) was defined when you created the agents in Foundry. From Python we simply *call* them — we don't need to re-define tools or instructions.

## Prerequisites

Before starting, make sure you have:

| What | Where to find it |
|------|------------------|
| **Foundry Project Endpoint** | Azure AI Foundry portal → your project → Overview → "Project endpoint" |
| **ZavaGroceriesInventoryAgent** name | Lab 1 — the agent name you typed when creating it |
| **SalesInsightsAgent** name | Lab 2 — the agent name you typed when creating it |
| Azure CLI authenticated | Run `az login` in your terminal if you haven't already |
| Python 3.10+ | Pre-installed in Codespaces |

## Step 1 — Install required packages

These are the official Microsoft SDKs for working with Foundry projects and agents.

In [ ]:
%pip install azure-ai-projects azure-identity openai --quiet

## Step 2 — Configure your project settings

Paste your Foundry project endpoint and agent names in the cell below. Replace each placeholder with your actual values.

> **Tip:** The agent names must match *exactly* what you typed in Foundry (including capitalization).

In [ ]:
# ── Paste your values here ──
PROJECT_ENDPOINT = "<paste-your-project-endpoint-here>"
INVENTORY_AGENT_NAME = "ZavaGroceriesInventoryAgent"
INSIGHTS_AGENT_NAME = "SalesInsightsAgent"

# Quick validation
assert not PROJECT_ENDPOINT.startswith("<"), "❌ Replace the PROJECT_ENDPOINT placeholder with your actual endpoint"
print("✅ Settings configured")
print(f"   Endpoint:        {PROJECT_ENDPOINT}")
print(f"   Inventory agent: {INVENTORY_AGENT_NAME}")
print(f"   Insights agent:  {INSIGHTS_AGENT_NAME}")

## Step 3 — Authenticate and connect to your Foundry project

This cell authenticates to Azure using your CLI credentials and creates two clients:
- **`project_client`** — manages project-level resources (agents, threads, etc.)
- **`openai_client`** — an OpenAI-compatible client that can call your Foundry agents

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()

# Store names in shorter variables for convenience
inventory_agent_name = INVENTORY_AGENT_NAME
insights_agent_name = INSIGHTS_AGENT_NAME

print("✅ Connected to Foundry project")

## Step 4 — Look up your agents

Let's verify that both agents exist in your Foundry project by listing all agents and finding ours by name.

In [ ]:
agents_client = project_client.agents

all_agents = list(agents_client.list_agents())
print(f"Found {len(all_agents)} agent(s) in your project:\n")
for a in all_agents:
    print(f"  \u2022 {a.name}  (model: {a.model})")

# Verify our two agents exist
inventory_agent = next((a for a in all_agents if a.name == inventory_agent_name), None)
insights_agent = next((a for a in all_agents if a.name == insights_agent_name), None)

assert inventory_agent, f"\u274c Agent '{inventory_agent_name}' not found \u2014 check spelling in your .env file"
assert insights_agent, f"\u274c Agent '{insights_agent_name}' not found \u2014 check spelling in your .env file"

print(f"\n\u2705 Both agents verified!")

## Step 5 — Create a helper function to call agents

We'll use the **OpenAI Responses API** to talk to our Foundry agents. This is the simplest approach — it handles tool execution (including MCP server calls) automatically on the server side.

The helper below wraps the API call so we can reuse it throughout the notebook.

In [ ]:
def run_agent(agent_name: str, prompt: str) -> str:
    """Send a prompt to a Foundry agent and return the response text."""
    response = openai_client.responses.create(
        input=prompt,
        extra_body={"agent": {"type": "agent_reference", "name": agent_name}},
    )
    return response.output_text

print("\u2705 run_agent() helper ready")

## Step 6 — Test the Inventory Agent

Let's send a prompt to **ZavaGroceriesInventoryAgent** and see it use its MCP tools to query live store data.

This agent has access to 10 CRUD tools (list, get, create, update, delete) for both daily financial records and hourly sales data.

In [ ]:
prompt = "List all daily financial records for Store-001. Summarize total chickens bought vs sold."

print(f"\U0001f4e4 Prompt: {prompt}\n")
print("\u23f3 Waiting for response (the agent is calling MCP tools behind the scenes)...\n")

result = run_agent(inventory_agent_name, prompt)

print(f"\U0001f4e5 Response from {inventory_agent_name}:\n")
print(result)

Let's try a second prompt to see the hourly sales data:

In [ ]:
prompt = "Show me the hourly sales breakdown for Store-001 for the most recent date. Which hours had the best sell-through?"

print(f"\U0001f4e4 Prompt: {prompt}\n")
print("\u23f3 Waiting for response...\n")

result = run_agent(inventory_agent_name, prompt)

print(f"\U0001f4e5 Response from {inventory_agent_name}:\n")
print(result)

## Step 7 — Test the Sales Insights Agent

Now let's test **SalesInsightsAgent**. This agent uses both MCP tools (for live data) and Azure AI Search (for news context) to produce a structured insights report with actionable recommendations.

In [ ]:
prompt = (
    "Analyze the complete sales performance for Store-001. "
    "Include ordering efficiency, cooking window analysis, "
    "relevant news context, and actionable recommendations."
)

print(f"\U0001f4e4 Prompt: {prompt}\n")
print("\u23f3 Waiting for response (this may take a moment \u2014 the agent is querying data + news)...\n")

result = run_agent(insights_agent_name, prompt)

print(f"\U0001f4e5 Response from {insights_agent_name}:\n")
print(result)

## Step 8 — Two-Agent Workflow: Insights → Inventory

Now for the key pattern: **agent chaining**. We'll run a two-step workflow:

```
+-------------------------+              +------------------------------+
| SalesInsightsAgent      |              | ZavaGroceriesInventoryAgent  |
|                         |  --output--> |                              |
| Analyze data &          |              | Review recommendations &     |
| produce action items    |              | check/apply to inventory     |
+-------------------------+              +------------------------------+
```

**How it works:**
1. We ask SalesInsightsAgent to generate specific recommendations
2. We capture its response text
3. We build a new prompt that includes those recommendations
4. We send that prompt to ZavaGroceriesInventoryAgent

This is the fundamental pattern behind all agent orchestration — one agent's output becomes another agent's input.

In [ ]:
# \u2500\u2500 Step A: Ask SalesInsightsAgent for recommendations \u2500\u2500
insights_prompt = (
    "Generate a full insights report for Store-001 with specific, numbered action items "
    "for improving ordering efficiency and cooking schedules. "
    "Focus on concrete changes with specific numbers."
)

print("=" * 70)
print("STEP A \u2014 Asking SalesInsightsAgent for recommendations")
print("=" * 70)
print(f"\n\U0001f4e4 Prompt:\n{insights_prompt}\n")
print("\u23f3 Generating insights...\n")

insights_output = run_agent(insights_agent_name, insights_prompt)

print(f"\U0001f4e5 SalesInsightsAgent response:\n")
print(insights_output)

In [ ]:
# \u2500\u2500 Step B: Feed recommendations to ZavaGroceriesInventoryAgent \u2500\u2500
inventory_prompt = (
    "The following recommendations were generated by our sales analytics system. "
    "Please review each action item and use your tools to check the current "
    "inventory data for Store-001, then confirm which changes can be applied:\n\n"
    f"{insights_output}"
)

print("\n" + "=" * 70)
print("STEP B \u2014 Passing recommendations to ZavaGroceriesInventoryAgent")
print("=" * 70)
print(f"\n\U0001f4e4 Prompt (first 300 chars):\n{inventory_prompt[:300]}...\n")
print("\u23f3 Inventory agent is reviewing recommendations...\n")

inventory_output = run_agent(inventory_agent_name, inventory_prompt)

print(f"\U0001f4e5 ZavaGroceriesInventoryAgent response:\n")
print(inventory_output)

## Validation Checklist

- [x] Connected to Foundry project from Python
- [x] Looked up both agents by name
- [x] Tested **ZavaGroceriesInventoryAgent** with live data queries
- [x] Tested **SalesInsightsAgent** with a full analysis prompt
- [x] Ran a **two-agent workflow** -- insights to inventory

## What you just built

You demonstrated the core pattern of **agent orchestration**: one agent's output becomes another agent's input. In production, this pattern scales to pipelines with many agents, conditional routing, and human-in-the-loop approvals.

## Troubleshooting

| Issue | Fix |
|-------|-----|
| `KeyError` on PROJECT_ENDPOINT | Replace the placeholder in Step 2 with your actual Foundry project endpoint |
| Agent not found assertion error | Check that agent names in Step 2 match exactly what you typed in Foundry (case-sensitive) |
| `DefaultAzureCredential` error | Run `az login` in your terminal |
| Timeout or empty response | The MCP server may be down -- verify it is running with `curl http://<IP>:8000/sse` |
| `ModuleNotFoundError` | Re-run Step 1 to install packages |